# Distributed Multi-Rank Stencil & Asynchronous Halo Exchange in MPI + CUDA (Python)

4-rank (2×2 Cartesian) anisotropic reaction–diffusion solver: `mpi4py` + `numba.cuda`, GPU halo pack/unpack kernels,
pinned host staging buffers, 16×16 shared-memory tiled stencil (18×18 tile), dual-stream compute/communication overlap.

**Runtime:** Google Colab, T4 GPU (Runtime → Change runtime type → T4 GPU), then Runtime → Run all.

### Cell 1 — Environment verification & mpi4py installation

In [7]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!which mpicc > /dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null)
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy "numba-cuda[cu12]"

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-b9215677-cb9a-672c-dbf4-a343db53485e, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



### Cell 2 — Student identity configuration

In [8]:

GITHUB_USERNAME = "mikazhan"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "You must set GITHUB_USERNAME to your real GitHub handle."
)
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: mikazhan


### Cell 3 — Distributed MPI + Numba CUDA solver (`%%writefile distributed_cuda_solver.py`)

In [9]:
%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import time
import hashlib
import platform
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, float64


TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15


PROC_DIMS = (2, 2)
REPORT_EVERY = 100
CFL_SAFETY = 0.8
KAPPA_MIN, KAPPA_MAX = 0.08, 0.24
N_PULSES = 3
PARITY_TOL = 1e-12
MANIFEST_PATH = "submission_manifest.json"


TAG_TO_UP, TAG_TO_DOWN, TAG_TO_LEFT, TAG_TO_RIGHT = 10, 11, 12, 13


def get_gpu_uuid() -> str:
    """Query active NVIDIA GPU UUID via nvidia-smi."""
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def get_gpu_name() -> str:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
            encoding="utf-8",
        )
        return out.strip().splitlines()[0].strip()
    except Exception:
        return "unknown"


def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    """Derive deterministic 64-bit integer seed bound to student GitHub handle and GPU UUID."""
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")).hexdigest()
    return int(digest[:16], 16)


def log0(ctx, msg):
    if ctx["rank"] == 0:
        print(msg, flush=True)


# ==============================================================================
# TASK 1: Distributed Cartesian Topology & Hardware Context Initialization
# ==============================================================================
def init_distributed_context(github_user: str):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks (2x2 grid), got {size}.")


    cuda.select_device(0)
    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    gpu_name = get_gpu_name() if rank == 0 else None
    gpu_name = comm.bcast(gpu_name, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)


    cart_comm = comm.Create_cart(dims=list(PROC_DIMS), periods=[False, False], reorder=False)
    cart_rank = cart_comm.Get_rank()
    row_rank, col_rank = cart_comm.Get_coords(cart_rank)
    nbr_up, nbr_down = cart_comm.Shift(0, 1)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    ny = GLOBAL_NY // PROC_DIMS[0]
    nx = GLOBAL_NX // PROC_DIMS[1]

    ctx = dict(
        comm=comm, cart_comm=cart_comm, rank=cart_rank, size=size,
        coords=(row_rank, col_rank),
        nbr_up=nbr_up, nbr_down=nbr_down, nbr_left=nbr_left, nbr_right=nbr_right,
        gpu_uuid=gpu_uuid, gpu_name=gpu_name, instance_seed=instance_seed,
        github_user=github_user, ny=ny, nx=nx,
    )

    def fmt(n):
        return "PROC_NULL" if n == MPI.PROC_NULL else str(n)

    for r in range(size):
        cart_comm.Barrier()
        if r == cart_rank:
            print(f"[rank {cart_rank}] coords=({row_rank},{col_rank}) local=({ny}+2)x({nx}+2) "
                  f"up={fmt(nbr_up)} down={fmt(nbr_down)} left={fmt(nbr_left)} right={fmt(nbr_right)}",
                  flush=True)
    cart_comm.Barrier()
    log0(ctx, f"[init] MPI ranks={size}, Cartesian dims={list(PROC_DIMS)}, periods=[False, False]")
    log0(ctx, f"[init] GPU={gpu_name} UUID={gpu_uuid}")
    log0(ctx, f"[init] GITHUB_USERNAME={github_user} INSTANCE_SEED={instance_seed} (0x{instance_seed:016x})")
    return ctx


# ==============================================================================
# TASK 2: Seeded Heterogeneous Subdomain Initialization
# ==============================================================================
def seeded_parameters(instance_seed: int):
    """All ranks derive identical physical parameters from INSTANCE_SEED (no communication)."""
    rng = np.random.default_rng(instance_seed)
    params = dict(
        kappa_phase=rng.uniform(0.0, 1.0, size=2),
        kappa_freq=rng.integers(1, 4, size=2),
        pulse_cx=rng.uniform(0.2, 0.8, size=N_PULSES),
        pulse_cy=rng.uniform(0.2, 0.8, size=N_PULSES),
        pulse_amp=rng.uniform(0.8, 1.5, size=N_PULSES),
        pulse_sigma=rng.uniform(0.05, 0.10, size=N_PULSES),
        src_cx=rng.uniform(0.3, 0.7),
        src_cy=rng.uniform(0.3, 0.7),
        src_amp=rng.uniform(0.01, 0.03),
        src_sigma=0.03,
    )
    return params


def physical_fields(X, Y, p):
    """Evaluate kappa(x,y), u0(x,y), S(x,y) on coordinate arrays (X = column dir j, Y = row dir i)."""
    mid = 0.5 * (KAPPA_MIN + KAPPA_MAX)
    amp = 0.5 * (KAPPA_MAX - KAPPA_MIN)
    kappa = mid + amp * (np.sin(2.0 * np.pi * (p["kappa_freq"][0] * X + p["kappa_phase"][0]))
                         * np.cos(2.0 * np.pi * (p["kappa_freq"][1] * Y + p["kappa_phase"][1])))
    kappa = np.clip(kappa, KAPPA_MIN, KAPPA_MAX)

    u0 = np.zeros_like(X)
    for k in range(N_PULSES):
        r2 = (X - p["pulse_cx"][k]) ** 2 + (Y - p["pulse_cy"][k]) ** 2
        u0 += p["pulse_amp"][k] * np.exp(-r2 / (2.0 * p["pulse_sigma"][k] ** 2))

    r2s = (X - p["src_cx"]) ** 2 + (Y - p["src_cy"]) ** 2
    S = p["src_amp"] * np.exp(-r2s / (2.0 * p["src_sigma"] ** 2))
    return kappa, u0, S


def initialize_subdomains(ctx):
    ny, nx = ctx["ny"], ctx["nx"]
    row_rank, col_rank = ctx["coords"]
    dx = 1.0 / (GLOBAL_NX + 1)
    dy = 1.0 / (GLOBAL_NY + 1)


    gi = row_rank * ny + np.arange(ny + 2, dtype=np.float64)
    gj = col_rank * nx + np.arange(nx + 2, dtype=np.float64)
    Y, X = np.meshgrid(gi * dy, gj * dx, indexing="ij")

    p = seeded_parameters(ctx["instance_seed"])
    kappa, u0, S = physical_fields(X, Y, p)


    u0[0, :] = 0.0
    u0[ny + 1, :] = 0.0
    u0[:, 0] = 0.0
    u0[:, nx + 1] = 0.0

    S[0, :] = S[ny + 1, :] = 0.0
    S[:, 0] = S[:, nx + 1] = 0.0

    u0 = np.ascontiguousarray(u0)
    kappa = np.ascontiguousarray(kappa)
    S = np.ascontiguousarray(S)

    local_kmax = np.array([kappa[1:ny + 1, 1:nx + 1].max()])
    local_kmin = np.array([kappa[1:ny + 1, 1:nx + 1].min()])
    kmax = np.zeros(1)
    kmin = np.zeros(1)
    ctx["cart_comm"].Allreduce(local_kmax, kmax, op=MPI.MAX)
    ctx["cart_comm"].Allreduce(local_kmin, kmin, op=MPI.MIN)
    kappa_max = float(kmax[0])
    dt_max = dx * dx / (4.0 * kappa_max)
    dt = CFL_SAFETY * dt_max

    phys = dict(dx=dx, dy=dy, dt=dt, dt_max=dt_max, kappa_max=kappa_max, kappa_min=float(kmin[0]),
                params={k: (v.tolist() if isinstance(v, np.ndarray) else float(v)) for k, v in p.items()})
    log0(ctx, f"[init] local padded shape={u0.shape}, dx=dy={dx:.6e}, kappa in [{phys['kappa_min']:.5f}, "
              f"{kappa_max:.5f}]")
    log0(ctx, f"[init] dt_max(CFL)={dt_max:.6e}, dt={dt:.6e} (safety {CFL_SAFETY})")
    return u0, kappa, S, phys


# ==============================================================================
# TASK 3: GPU Halo Pack/Unpack Kernels & Pinned MPI Exchange
# ==============================================================================

@cuda.jit
def pack_halos_kernel(u, send_buf, ny, nx):
    k = cuda.grid(1)
    if k < nx:
        send_buf[k] = u[1, k + 1]
        send_buf[nx + k] = u[ny, k + 1]
    if k < ny:
        send_buf[2 * nx + k] = u[k + 1, 1]
        send_buf[2 * nx + ny + k] = u[k + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, recv_buf, ny, nx):
    k = cuda.grid(1)
    if k < nx:
        u[0, k + 1] = recv_buf[k]
        u[ny + 1, k + 1] = recv_buf[nx + k]
    if k < ny:
        u[k + 1, 0] = recv_buf[2 * nx + k]
        u[k + 1, nx + 1] = recv_buf[2 * nx + ny + k]

class HaloBuffers:
    """Device 1D send/recv vectors + page-locked (pinned) host staging buffers."""

    def __init__(self, ny, nx):
        self.ny, self.nx = ny, nx
        n = 2 * nx + 2 * ny
        self.n = n
        self.d_send = cuda.device_array(n, dtype=np.float64)
        self.d_recv = cuda.to_device(np.zeros(n, dtype=np.float64))
        self.h_send = cuda.pinned_array(n, dtype=np.float64)
        self.h_recv = cuda.pinned_array(n, dtype=np.float64)
        self.h_send[:] = 0.0

        self.h_recv[:] = 0.0
        self.threads = 256
        self.blocks = (max(nx, ny) + self.threads - 1) // self.threads


    def segs(self, buf):
        nx, ny = self.nx, self.ny
        return dict(top=buf[0:nx], bottom=buf[nx:2 * nx],
                    left=buf[2 * nx:2 * nx + ny], right=buf[2 * nx + ny:2 * nx + 2 * ny])


def exchange_halos_async(ctx, bufs, d_u, stream_boundary):
    """Stage A + B(MPI part): GPU pack -> async D2H into pinned -> post Irecv/Isend.
    Returns MPI requests; the caller overlaps compute and then calls finish_halos()."""
    ny, nx = bufs.ny, bufs.nx
    cart = ctx["cart_comm"]

    pack_halos_kernel[bufs.blocks, bufs.threads, stream_boundary](d_u, bufs.d_send, ny, nx)
    bufs.d_send.copy_to_host(bufs.h_send, stream=stream_boundary)
    stream_boundary.synchronize()

    s = bufs.segs(bufs.h_send)
    r = bufs.segs(bufs.h_recv)
    reqs = [

        cart.Irecv([r["top"], MPI.DOUBLE], source=ctx["nbr_up"], tag=TAG_TO_DOWN),
        cart.Irecv([r["bottom"], MPI.DOUBLE], source=ctx["nbr_down"], tag=TAG_TO_UP),
        cart.Irecv([r["left"], MPI.DOUBLE], source=ctx["nbr_left"], tag=TAG_TO_RIGHT),
        cart.Irecv([r["right"], MPI.DOUBLE], source=ctx["nbr_right"], tag=TAG_TO_LEFT),

        cart.Isend([s["top"], MPI.DOUBLE], dest=ctx["nbr_up"], tag=TAG_TO_UP),
        cart.Isend([s["bottom"], MPI.DOUBLE], dest=ctx["nbr_down"], tag=TAG_TO_DOWN),
        cart.Isend([s["left"], MPI.DOUBLE], dest=ctx["nbr_left"], tag=TAG_TO_LEFT),
        cart.Isend([s["right"], MPI.DOUBLE], dest=ctx["nbr_right"], tag=TAG_TO_RIGHT),
    ]
    return reqs


def finish_halos(bufs, reqs, d_u, stream_boundary):
    """Stage C (first half): Waitall -> async H2D -> GPU unpack into ghost rows/cols 0 and n+1."""
    MPI.Request.Waitall(reqs)
    bufs.d_recv.copy_to_device(bufs.h_recv, stream=stream_boundary)
    unpack_halos_kernel[bufs.blocks, bufs.threads, stream_boundary](d_u, bufs.d_recv, bufs.ny, bufs.nx)


# ==============================================================================
# TASK 4: 2D Shared-Memory Tiled Stencil & Dual-Stream Orchestrator
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, u_new, kappa, src, r_start, r_end, c_start, c_end,
                         dt, inv_dx2, inv_dy2, lam):
    smem = cuda.shared.array(shape=(SHARED_DIM, SHARED_DIM), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = r_start + cuda.blockIdx.y * TILE_DIM + ty   # global (local-array) row
    j = c_start + cuda.blockIdx.x * TILE_DIM + tx   # global (local-array) column
    sy = ty + 1
    sx = tx + 1

    lo_r = r_start - 1
    hi_r = r_end + 1
    lo_c = c_start - 1
    hi_c = c_end + 1


    if i <= hi_r and j <= hi_c:
        smem[sy, sx] = u[i, j]

    if ty == 0 and i - 1 >= lo_r and j <= hi_c:
        smem[0, sx] = u[i - 1, j]
    if ty == TILE_DIM - 1 and i + 1 <= hi_r and j <= hi_c:
        smem[SHARED_DIM - 1, sx] = u[i + 1, j]
    if tx == 0 and j - 1 >= lo_c and i <= hi_r:
        smem[sy, 0] = u[i, j - 1]
    if tx == TILE_DIM - 1 and j + 1 <= hi_c and i <= hi_r:
        smem[sy, SHARED_DIM - 1] = u[i, j + 1]

    cuda.syncthreads()

    if i <= r_end and j <= c_end:
        uc = smem[sy, sx]
        u_im = smem[sy - 1, sx]
        u_ip = smem[sy + 1, sx]
        u_jm = smem[sy, sx - 1]
        u_jp = smem[sy, sx + 1]
        kc = kappa[i, j]
        k_ip = 0.5 * (kappa[i + 1, j] + kc)
        k_im = 0.5 * (kappa[i - 1, j] + kc)
        k_jp = 0.5 * (kappa[i, j + 1] + kc)
        k_jm = 0.5 * (kappa[i, j - 1] + kc)
        div_i = (k_ip * (u_ip - uc) - k_im * (uc - u_im)) * inv_dx2
        div_j = (k_jp * (u_jp - uc) - k_jm * (uc - u_jm)) * inv_dy2
        u_new[i, j] = uc + dt * (div_i + div_j - lam * uc * uc * uc + src[i, j])


def launch_stencil(d_u, d_un, d_k, d_s, box, stream, phys):
    r0, r1, c0, c1 = box
    grid = ((c1 - c0 + 1 + TILE_DIM - 1) // TILE_DIM, (r1 - r0 + 1 + TILE_DIM - 1) // TILE_DIM)
    tiled_stencil_kernel[grid, (TILE_DIM, TILE_DIM), stream](
        d_u, d_un, d_k, d_s, r0, r1, c0, c1,
        phys["dt"], 1.0 / (phys["dx"] ** 2), 1.0 / (phys["dy"] ** 2), LAMBDA_DAMP)


def region_boxes(ny, nx):
    full = (1, ny, 1, nx)
    interior = (2, ny - 1, 2, nx - 1)
    ring = [
        (1, 1, 1, nx),
        (ny, ny, 1, nx),
        (2, ny - 1, 1, 1),
        (2, ny - 1, nx, nx),
    ]
    return full, interior, ring


def step_serialized(ctx, st, bufs, phys):
    """Reference: blocking-style halo exchange, then one full-domain stencil, single stream."""
    s = st["stream_boundary"]
    reqs = exchange_halos_async(ctx, bufs, st["u"], s)
    finish_halos(bufs, reqs, st["u"], s)
    launch_stencil(st["u"], st["un"], st["k"], st["s"], st["full"], s, phys)
    s.synchronize()
    st["u"], st["un"] = st["un"], st["u"]


def step_overlapped(ctx, st, bufs, phys):
    sb, si = st["stream_boundary"], st["stream_interior"]

    reqs = exchange_halos_async(ctx, bufs, st["u"], sb)

    launch_stencil(st["u"], st["un"], st["k"], st["s"], st["interior"], si, phys)

    finish_halos(bufs, reqs, st["u"], sb)
    for box in st["ring"]:
        launch_stencil(st["u"], st["un"], st["k"], st["s"], box, sb, phys)
    sb.synchronize()
    si.synchronize()
    st["u"], st["un"] = st["un"], st["u"]


# ==============================================================================
# TASK 5: Distributed Reductions, PTX Audit & Comparative Benchmark
# ==============================================================================
def global_diagnostics(ctx, st, phys, h_prev):
    ny, nx = ctx["ny"], ctx["nx"]
    h_u = st["u"].copy_to_host()
    interior = h_u[1:ny + 1, 1:nx + 1]
    loc = np.array([
        interior.sum() * phys["dx"] * phys["dy"],
        0.0 if h_prev is None else float(((interior - h_prev) ** 2).sum()),
        float((interior ** 2).sum()),
        float(np.abs(interior).max()),
    ])
    glob_sum = np.zeros(3)
    ctx["cart_comm"].Allreduce(loc[:3].copy(), glob_sum, op=MPI.SUM)
    glob_max = np.zeros(1)
    ctx["cart_comm"].Allreduce(loc[3:4].copy(), glob_max, op=MPI.MAX)
    energy = float(glob_sum[0])
    l2_residual = math.sqrt(glob_sum[1] * phys["dx"] * phys["dy"])  # ||u^n - u^(n-1)||_L2
    l2_norm = math.sqrt(glob_sum[2] * phys["dx"] * phys["dy"])
    return energy, l2_residual, l2_norm, float(glob_max[0]), interior.copy()


def make_state(u0, kappa, S, ctx):
    ny, nx = ctx["ny"], ctx["nx"]
    full, interior, ring = region_boxes(ny, nx)
    return dict(
        u=cuda.to_device(u0), un=cuda.to_device(u0),
        k=cuda.to_device(kappa), s=cuda.to_device(S),
        stream_boundary=cuda.stream(), stream_interior=cuda.stream(),
        full=full, interior=interior, ring=ring,
    )


def run_mode(ctx, mode, u0, kappa, S, phys, bufs, steps):
    st = make_state(u0, kappa, S, ctx)
    step_fn = step_serialized if mode == "serialized" else step_overlapped
    trace = []
    h_prev = None
    step_time = 0.0
    ev_total_ms = 0.0

    e, r, n, m, h_prev = global_diagnostics(ctx, st, phys, None)
    trace.append(dict(step=0, energy=e, l2_residual=r, l2_norm=n, max_abs_u=m))
    log0(ctx, f"[{mode:>10}] step {0:4d}  E={e:.12e}  dU_L2=n/a        ||u||2={n:.6e}  max|u|={m:.6f}")

    for start in range(0, steps, REPORT_EVERY):
        chunk = min(REPORT_EVERY, steps - start)
        cuda.synchronize()
        ctx["cart_comm"].Barrier()
        ev0, ev1 = cuda.event(), cuda.event()
        t0 = MPI.Wtime()
        ev0.record()
        for _ in range(chunk):
            step_fn(ctx, st, bufs, phys)
        ev1.record()
        ev1.synchronize()
        cuda.synchronize()
        step_time += MPI.Wtime() - t0
        try:
            ev_total_ms += cuda.event_elapsed_time(ev0, ev1)
        except Exception:
            pass

        e, r, n, m, h_cur = global_diagnostics(ctx, st, phys, h_prev)
        trace.append(dict(step=start + chunk, energy=e, l2_residual=r, l2_norm=n, max_abs_u=m))
        log0(ctx, f"[{mode:>10}] step {start + chunk:4d}  E={e:.12e}  dU_L2({REPORT_EVERY} steps)={r:.3e}  "
                  f"||u||2={n:.6e}  max|u|={m:.6f}")
        h_prev = h_cur

    wall_ms = np.array([step_time * 1e3])
    evm = np.array([ev_total_ms])
    wall_max = np.zeros(1)
    ev_max = np.zeros(1)
    ctx["cart_comm"].Allreduce(wall_ms, wall_max, op=MPI.MAX)
    ctx["cart_comm"].Allreduce(evm, ev_max, op=MPI.MAX)
    final_local = st["u"].copy_to_host()
    return dict(trace=trace, wall_ms=float(wall_max[0]), event_ms=float(ev_max[0]), final=final_local)


def ptx_audit():
    try:
        asm = tiled_stencil_kernel.inspect_asm()
    except Exception as exc:
        return dict(available=False, error=repr(exc), st_shared_f64=0, ld_shared_f64=0,
                    bar_sync=0, passed=False)
    if isinstance(asm, dict):
        ptx = "\n".join(asm.values())
    else:
        ptx = str(asm)
    import re
    st_count = len(re.findall(r"\bst\.shared\.f64\b", ptx))
    ld_count = len(re.findall(r"\bld\.shared\.f64\b", ptx))
    st_any = len(re.findall(r"\bst\.shared\.", ptx))
    ld_any = len(re.findall(r"\bld\.shared\.", ptx))
    bar = len(re.findall(r"\bbar\.sync\b", ptx)) + len(re.findall(r"\bbarrier\.sync\b", ptx))
    return dict(available=True, st_shared_f64=st_count, ld_shared_f64=ld_count,
                st_shared_any=st_any, ld_shared_any=ld_any, bar_sync=bar,
                ptx_lines=len(ptx.splitlines()),
                passed=bool(st_any > 0 and ld_any > 0 and bar > 0))


def warmup(ctx, u0, kappa, S, phys, bufs):
    """JIT-compile every kernel once (excluded from timing)."""
    st = make_state(u0, kappa, S, ctx)
    step_overlapped(ctx, st, bufs, phys)
    step_serialized(ctx, st, bufs, phys)
    cuda.synchronize()
    ctx["cart_comm"].Barrier()


def run_solver(github_user: str, steps: int = STEPS, write_manifest: bool = True):
    ctx = init_distributed_context(github_user)
    u0, kappa, S, phys = initialize_subdomains(ctx)
    bufs = HaloBuffers(ctx["ny"], ctx["nx"])

    warmup(ctx, u0, kappa, S, phys, bufs)
    log0(ctx, f"[run] kernels compiled; running {steps} steps per mode")

    ser = run_mode(ctx, "serialized", u0, kappa, S, phys, bufs, steps)
    ovl = run_mode(ctx, "overlapped", u0, kappa, S, phys, bufs, steps)


    loc_diff = np.array([float(np.abs(ser["final"] - ovl["final"]).max())])
    glob_diff = np.zeros(1)
    ctx["cart_comm"].Allreduce(loc_diff, glob_diff, op=MPI.MAX)
    parity_max_diff = float(glob_diff[0])
    parity_ok = parity_max_diff < PARITY_TOL


    topo = np.zeros((ctx["size"], 6), dtype=np.int64)
    topo[ctx["rank"]] = [ctx["coords"][0], ctx["coords"][1],
                         ctx["nbr_up"], ctx["nbr_down"], ctx["nbr_left"], ctx["nbr_right"]]
    topo_all = np.zeros_like(topo)
    ctx["cart_comm"].Allreduce(topo, topo_all, op=MPI.SUM)

    audit = ptx_audit() if ctx["rank"] == 0 else None
    audit = ctx["cart_comm"].bcast(audit, root=0)

    energies = [t["energy"] for t in ovl["trace"]]
    monotone = all(energies[k + 1] <= energies[k] for k in range(len(energies) - 1))
    speedup = ser["wall_ms"] / ovl["wall_ms"] if ovl["wall_ms"] > 0 else float("nan")

    if ctx["rank"] == 0:
        print("=" * 78)
        print(f"[bench] serialized_ms = {ser['wall_ms']:.3f}  (CUDA events: {ser['event_ms']:.3f} ms)")
        print(f"[bench] overlapped_ms = {ovl['wall_ms']:.3f}  (CUDA events: {ovl['event_ms']:.3f} ms)")
        print(f"[bench] overlap speedup = {speedup:.4f}x")
        print(f"[parity] max |u_serialized - u_overlapped| = {parity_max_diff:.3e} -> "
              f"{'PASS' if parity_ok else 'FAIL'} (tol {PARITY_TOL:g})")
        print(f"[ptx] st.shared.f64={audit['st_shared_f64']} ld.shared.f64={audit['ld_shared_f64']} "
              f"bar.sync={audit['bar_sync']} -> {'PASS' if audit['passed'] else 'FAIL'}")
        print(f"[energy] monotonic non-increasing over reported steps: {monotone}")
        print("=" * 78)

    manifest = None
    if ctx["rank"] == 0:
        try:
            with open(os.path.abspath(__file__), "rb") as fh:
                src_hash = hashlib.sha256(fh.read()).hexdigest()
        except Exception:
            src_hash = None
        import numba
        manifest = dict(
            github_username=github_user,
            gpu_uuid=ctx["gpu_uuid"],
            gpu_name=ctx["gpu_name"],
            instance_seed=str(ctx["instance_seed"]),
            instance_seed_hex=f"0x{ctx['instance_seed']:016x}",
            mpi_ranks=ctx["size"],
            cart_dims=list(PROC_DIMS),
            cart_periods=[False, False],
            topology=[dict(rank=r, coords=[int(topo_all[r, 0]), int(topo_all[r, 1])],
                           up=int(topo_all[r, 2]), down=int(topo_all[r, 3]),
                           left=int(topo_all[r, 4]), right=int(topo_all[r, 5]))
                      for r in range(ctx["size"])],
            mpi_proc_null=int(MPI.PROC_NULL),
            global_grid=[GLOBAL_NY, GLOBAL_NX],
            local_padded_shape=[ctx["ny"] + 2, ctx["nx"] + 2],
            tile_dim=TILE_DIM, shared_dim=SHARED_DIM,
            steps=steps, lambda_damp=LAMBDA_DAMP,
            dx=phys["dx"], dy=phys["dy"], dt=phys["dt"], dt_max_cfl=phys["dt_max"],
            kappa_max=phys["kappa_max"], kappa_min=phys["kappa_min"],
            seeded_params=phys["params"],
            energy_trace=ovl["trace"],
            energy_trace_serialized=ser["trace"],
            energy_monotonic=monotone,
            serialized_ms=ser["wall_ms"], overlapped_ms=ovl["wall_ms"],
            serialized_event_ms=ser["event_ms"], overlapped_event_ms=ovl["event_ms"],
            overlap_speedup=speedup,
            parity_max_diff=parity_max_diff, parity_passed=parity_ok,
            ptx_audit=audit, ptx_audit_passed=audit["passed"],
            tasks_completed=[1, 2, 3, 4, 5],
            versions=dict(python=platform.python_version(), numpy=np.__version__,
                          numba=numba.__version__, mpi=MPI.Get_library_version().splitlines()[0]),
            solver_sha256=src_hash,
            timestamp_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        )
        if write_manifest:
            with open(MANIFEST_PATH, "w") as fh:
                json.dump(manifest, fh, indent=2)
            print(f"[manifest] wrote {MANIFEST_PATH}")
    return dict(ctx=ctx, serialized=ser, overlapped=ovl, parity=parity_max_diff,
                audit=audit, manifest=manifest, phys=phys)


if __name__ == "__main__":
    if len(sys.argv) < 2 or not sys.argv[1].strip():
        raise SystemExit("usage: mpirun -np 4 python3 distributed_cuda_solver.py <github_username>")
    run_solver(sys.argv[1].strip())

Overwriting distributed_cuda_solver.py


### Cell 4 — Multi-rank MPI + CUDA execution launcher

In [10]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")

[rank 0] coords=(0,0) local=(256+2)x(256+2) up=PROC_NULL down=2 left=PROC_NULL right=1
[rank 1] coords=(0,1) local=(256+2)x(256+2) up=PROC_NULL down=3 left=0 right=PROC_NULL
[rank 2] coords=(1,0) local=(256+2)x(256+2) up=0 down=PROC_NULL left=PROC_NULL right=3
[rank 3] coords=(1,1) local=(256+2)x(256+2) up=1 down=PROC_NULL left=2 right=PROC_NULL
[init] MPI ranks=4, Cartesian dims=[2, 2], periods=[False, False]
[init] GPU=Tesla T4, 7.5 UUID=GPU-b9215677-cb9a-672c-dbf4-a343db53485e
[init] GITHUB_USERNAME=mikazhan INSTANCE_SEED=8143002696515884286 (0x7101c1cf1ae094fe)
[init] local padded shape=(258, 258), dx=dy=1.949318e-03, kappa in [0.08000, 0.24000]
[init] dt_max(CFL)=3.958173e-06, dt=3.166538e-06 (safety 0.8)
[run] kernels compiled; running 400 steps per mode
[serialized] step    0  E=8.129855280073e-02  dU_L2=n/a        ||u||2=2.161839e-01  max|u|=1.350258
[serialized] step  100  E=8.129513507954e-02  dU_L2(100 steps)=3.830e-03  ||u||2=2.135831e-01  max|u|=1.323026
[serialized] step 

### Telemetry summary for the analytical answers (read from `submission_manifest.json`)

In [11]:
import json
m = json.load(open("submission_manifest.json"))
dx = m["dx"]; ny, nx = m["local_padded_shape"][0] - 2, m["local_padded_shape"][1] - 2
print(f"GitHub user        : {m['github_username']}")
print(f"GPU                : {m['gpu_name']}  {m['gpu_uuid']}")
print(f"INSTANCE_SEED      : {m['instance_seed']} ({m['instance_seed_hex']})")
print(f"MPI ranks / dims   : {m['mpi_ranks']} / {m['cart_dims']}")
for t in m["topology"]:
    print(f"   rank {t['rank']} coords={t['coords']} up={t['up']} down={t['down']} left={t['left']} right={t['right']}  (PROC_NULL={m['mpi_proc_null']})")
print("--- Q1 ---")
span = ((ny - 1) * (nx + 2) + 1) * 8
print(f"column payload = {ny*8} B ; contiguous span u[1,1]..u[{ny},1] = {span} B ; extra = {span - ny*8} B per column")
print(f"extra per step per rank (2 cols D2H + 2 ghost cols H2D) = {4*(span - ny*8)} B ; over {m['steps']} steps = {4*(span - ny*8)*m['steps']/1e6:.1f} MB/rank")
print("--- Q3 ---")
a = m["ptx_audit"]
print(f"st.shared.f64={a['st_shared_f64']}  ld.shared.f64={a['ld_shared_f64']}  bar.sync={a['bar_sync']}  passed={m['ptx_audit_passed']}")
print("--- Q4 ---")
print(f"serialized_ms={m['serialized_ms']:.3f}  overlapped_ms={m['overlapped_ms']:.3f}  speedup={m['overlap_speedup']:.4f}x")
print(f"CUDA-event ms: serialized={m['serialized_event_ms']:.3f} overlapped={m['overlapped_event_ms']:.3f}")
print("--- Q6 ---")
print(f"kappa_max={m['kappa_max']:.6f}  dx={dx:.6e}  dx^2={dx*dx:.6e}")
print(f"dt_max = dx^2/(4 kappa_max) = {m['dt_max_cfl']:.6e} ; dt used = {m['dt']:.6e} ; ratio = {m['dt']/m['dt_max_cfl']:.3f}")
for t in m["energy_trace"]:
    print(f"   step {t['step']:4d}: E = {t['energy']:.12e}   ||u||_L2 = {t['l2_norm']:.6e}   max|u| = {t['max_abs_u']:.6f}")
print(f"energy monotonic: {m['energy_monotonic']} ; parity_max_diff = {m['parity_max_diff']:.3e}")

GitHub user        : mikazhan
GPU                : Tesla T4, 7.5  GPU-b9215677-cb9a-672c-dbf4-a343db53485e
INSTANCE_SEED      : 8143002696515884286 (0x7101c1cf1ae094fe)
MPI ranks / dims   : 4 / [2, 2]
   rank 0 coords=[0, 0] up=-2 down=2 left=-2 right=1  (PROC_NULL=-2)
   rank 1 coords=[0, 1] up=-2 down=3 left=0 right=-2  (PROC_NULL=-2)
   rank 2 coords=[1, 0] up=0 down=-2 left=-2 right=3  (PROC_NULL=-2)
   rank 3 coords=[1, 1] up=1 down=-2 left=2 right=-2  (PROC_NULL=-2)
--- Q1 ---
column payload = 2048 B ; contiguous span u[1,1]..u[256,1] = 526328 B ; extra = 524280 B per column
extra per step per rank (2 cols D2H + 2 ghost cols H2D) = 2097120 B ; over 400 steps = 838.8 MB/rank
--- Q3 ---
st.shared.f64=5  ld.shared.f64=5  bar.sync=1  passed=True
--- Q4 ---
serialized_ms=2411.568  overlapped_ms=3374.394  speedup=0.7147x
CUDA-event ms: serialized=2410.783 overlapped=3374.217
--- Q6 ---
kappa_max=0.240000  dx=1.949318e-03  dx^2=3.799840e-06
dt_max = dx^2/(4 kappa_max) = 3.958173e-06 ; d

### Analytical & Forensic Engineering Responses

> All numbers below come from my run: GitHub `mikazhan`, Tesla T4 (CC 7.5) `GPU-b9215677-cb9a-672c-dbf4-a343db53485e`, `INSTANCE_SEED = 8143002696515884286 (0x7101c1cf1ae094fe)`.

- **Question 1 (Memory Coalescing & Strided Column Packing):**
  The local field is a C-contiguous `(258, 258)` `float64` array, so element `u[i, j]` lives at byte offset `8·(258·i + j)`. A row (`u[1, 1:257]`) is 256 consecutive doubles = 2 048 contiguous bytes, but successive elements of a column (`u[1:257, 1]`, `u[1:257, 256]`) are `258·8 = 2 064 B` apart. A DMA copy (`cudaMemcpyAsync`) moves one *contiguous* byte range, so it cannot pick out 256 isolated 8-byte words separated by 2 056-byte gaps. At warp level the same stride is the problem: if 32 threads of a warp read 32 consecutive column elements, each lands in a different 32-byte sector / 128-byte line, giving 32 memory transactions per warp instead of 2 (≈16× wasted bandwidth). My `pack_halos_kernel` uses a 1D grid where thread `k` reads `u[k+1, 1]` (strided, unavoidable once) but **writes** `send_buf[2nx + k]` contiguously, so the PCIe transfer that follows is one dense 8 192-byte (4 × 256 × 8) copy.
  If one instead copied the smallest contiguous span that contains a column, `u[1,1] … u[256,1]`, that is `(255·258 + 1)·8 = 526 328 B` for `2 048 B` of payload → **524 280 extra bytes per column** (≈257× overhead). Per step and rank that is 2 columns D2H + 2 ghost columns H2D = **2 097 120 extra bytes**, i.e. ≈ **838.8 MB per rank over 400 steps** (≈3.4 GB across 4 ranks), versus 8 192 B × 2 directions per step with packing (the telemetry cell printed exactly these values: 2048 B payload, 526 328 B span, 524 280 B extra per column, 2 097 120 B per step, 838.8 MB/rank).

- **Question 2 (Shared-Memory Tile Indexing & Bank Conflict Analysis):**
  With `sy = ty+1`, `sx = tx+1`:
  - Thread `(tx=0, ty=0)` **writes** `smem[1,1]` (center `u[i,j]`), `smem[0,1]` (top apron `u[i−1,j]`, because `ty==0`) and `smem[1,0]` (left apron `u[i,j−1]`, because `tx==0`). After `syncthreads()` it **reads** `smem[1,1]`, `smem[0,1]`, `smem[2,1]`, `smem[1,0]`, `smem[1,2]`.
  - Thread `(tx=15, ty=15)` **writes** `smem[16,16]`, `smem[17,16]` (bottom apron, `ty==15`) and `smem[16,17]` (right apron, `tx==15`); it **reads** `smem[16,16]`, `smem[15,16]`, `smem[17,16]`, `smem[16,15]`, `smem[16,17]`.
  Each `float64` occupies two 4-byte banks; `smem[r,c]` starts at word `2·(18r + c)`, bank `(36r + 2c) mod 32`. E.g. `smem[1,1]` → banks 6–7, `smem[16,16]` → banks 0–1, `smem[17,16]` → banks 4–5.
  **Row access (what the kernel does):** threads are linearised as `tx + 16·ty`, so a warp is two tile rows and each half-warp is one row of 16 consecutive doubles = 32 consecutive words = all 32 banks exactly once. On CC 7.5 a 64-bit shared access by a warp is split into two 128-byte phases (one per half-warp), so row loads are **conflict-free** (2 wavefronts, which is the minimum for 256 bytes, not a conflict).
  **Column access:** the row pitch is 18 doubles = 36 words ≡ 4 (mod 32). Sixteen threads walking down a column hit bank pairs `4k mod 32`, which repeat after 8 rows → **2-way bank conflict**. The kernel only does column-wise accesses for the left/right apron (2 threads per warp, banks 4 apart → no conflict), so in practice the stencil has no shared-memory bank conflicts. (Without the +2 apron, a pitch of 16 doubles = 32 words would make a column access a 16-way conflict.)

- **Question 3 (PTX Assembly Audit & Warp Synchronization):**
  My run's `tiled_stencil_kernel.inspect_asm()` reports **st.shared.f64 = 5, ld.shared.f64 = 5, bar.sync = 1** (`ptx_audit_passed = true`). Interpretation: five stores = 1 center + 4 conditional apron stores; five loads = the 5-point stencil `c, i±1, j±1`; one barrier = the single `cuda.syncthreads()`.
  The 16×16 block is 8 warps; warp *w* owns tile rows `ty = 2w, 2w+1`. A thread in row `ty` reads `smem[ty, ·]` and `smem[ty+2, ·]`, i.e. rows stored **by a different warp** (e.g. warp 0, `ty=1`, reads `smem[3, sx]`, written by warp 1), and the apron rows `smem[0,·]`/`smem[17,·]` are written only by warps 0 and 7. Warps are scheduled independently, so without `bar.sync` there is a cross-warp **read-after-write (RAW) hazard**: a warp may execute its `ld.shared` before the producing warp's `st.shared`, reading stale data left by a previous block on that SM. Results become non-deterministic, the serialized/overlapped parity check (`parity_max_diff = 0.000e+00` in my run) would fail, and errors would appear in a pattern of 2-row stripes. `bar.sync 0` makes all 256 threads' shared stores visible before any stencil load.

- **Question 4 (Multi-Process Context Sharing on a Single Physical GPU):**
  `mpirun -np 4` starts four OS processes; each calls `cuda.select_device(0)` and creates its **own CUDA context** on the single T4. Without MPS (not enabled on Colab), contexts are **time-sliced**: the GPU scheduler runs kernels of one context at a time on the 40 SMs and switches between contexts, so kernels from different ranks never execute concurrently, and every context switch adds overhead. The copy (DMA) engines are also shared; transfers of different processes are serialized through the same PCIe link. Inside one context my two streams *can* overlap (interior kernel on `stream_interior` while the boundary H2D/unpack runs on `stream_boundary`). MPI traffic itself goes through host shared memory (OpenMPI `vader`/`sm` BTL) using the pinned buffers.
  Measured (400 steps, max over ranks): **serialized_ms = 2411.568, overlapped_ms = 3374.394 → speedup 0.7147×** (CUDA events: 2410.783 / 3374.217 ms), i.e. 6.03 ms vs 8.44 ms per step — the overlapped version is ≈2.41 ms/step **slower**. Interpretation: per step the communication is tiny (4 × 2 KB through shared memory, a few µs) and the interior kernel on a 256×256 block is only tens of µs, so there is little latency to hide, while the overlapped version issues 5 stencil launches instead of 1 plus extra stream synchronisations — Python/launch overhead and context time-slicing dominate, so in my run the "overlap" is a net loss (0.71×): the extra launches and synchronisations cost more than the few µs of MPI latency they could hide, and the interior kernel of one rank cannot truly run concurrently with kernels of the other three ranks. On four dedicated GPUs over InfiniBand each rank owns its GPU (no time-slicing), messages cross a real network with µs-level latency and limited bandwidth (and, at production sizes, far larger halos), so posting `Isend/Irecv` and computing the deep interior meanwhile hides most of that latency, and overlap gives a clear speedup. A previous run on the same T4 gave 927 / 1392 ms (0.67×): absolute times vary ~2.5× between sessions on the shared Colab GPU, but the ratio stays consistent.

- **Question 5 (Point-to-Point Message Deadlock & MPI.PROC_NULL Semantics):**
  A blocking `MPI_Send` may return only when the buffer can be reused. OpenMPI uses an **eager protocol** for messages below the transport's eager limit (≈4 KB for the shared-memory `vader` BTL, 64 KB for TCP): the data is copied into a system buffer at the receiver and `Send` returns without a matching `Recv`. Above the limit it switches to the **rendezvous protocol**: the sender sends a request-to-send and blocks until the receiver posts the matching `Recv`. If every rank executes the same fixed order "Send up, Send down, Send left, Send right, then Recv …", then rank (0,0) blocks in `Send(down → rank 2)` while rank (1,0) blocks in `Send(up → rank 0)`; neither reaches its `Recv`, forming a cycle of waits → **deadlock**. In this lab a halo is 256 doubles = 2 048 B < 4 KB, so it would *happen to work* eagerly; with `nx ≥ 513` per rank (> 4 096 B) or a different transport limit it hangs — a latent bug that appears only on large grids. Non-blocking `Irecv`/`Isend` + `Waitall` (my implementation) avoids it because all operations are posted before anyone waits.
  `MPI.PROC_NULL` is a valid dummy rank: a send to it completes immediately with no data, and a receive from it completes immediately leaving the buffer unchanged (count 0). Rank (0,0) has `nbr_up = nbr_left = PROC_NULL` (see topology printout), yet runs exactly the same code as interior ranks — no `if` branches for corners/edges. Because its recv slots for those sides are never written, they keep the initial 0.0 and the unpack kernel writes 0 into the ghost row/column, which is exactly the homogeneous Dirichlet condition `u = 0`.

- **Question 6 (Explicit CFL Stability & Energy Conservation Proof):**
  Δx = Δy = 1/513 = 1.949318×10⁻³, Δx² = 3.799840×10⁻⁶. My run recorded κ_max = **0.240000** (κ ∈ [0.08000, 0.24000]), so
  Δt_max = Δx² / (4·κ_max) = 3.799840×10⁻⁶ / 0.96 = **3.958173×10⁻⁶**. The solver used **Δt = 3.166538×10⁻⁶** (ratio 0.800), i.e. 80 % of the limit; for the 2-D 5-point scheme with Δx = Δy the amplification factor of the highest-frequency mode is 1 − 8κΔt/Δx² ≥ 1 − 0.8·2 = −0.6, so |G| ≤ 1 and the scheme is stable (the cubic term −λu³ only adds damping since λΔt·u² ≪ 1).
  Global energy E = Σ u·Δx·Δy from `cart_comm.Allreduce(op=MPI.SUM)`:

  | step | E (Allreduce SUM) | ΔE vs previous | max\|u\| |
  |---|---|---|---|
  | 0 | 8.129855280073e-02 | — | 1.350258 |
  | 100 | 8.129513507954e-02 | −3.418e-06 | 1.323026 |
  | 200 | 8.129208625973e-02 | −3.049e-06 | 1.296867 |
  | 300 | 8.128897216970e-02 | −3.114e-06 | 1.271719 |
  | 400 | 8.128572431111e-02 | −3.248e-06 | 1.247523 |

  The sequence decreases monotonically (`energy_monotonic = True`, total −1.58×10⁻⁴ relative over 400 steps); ‖u‖₂ also falls 0.2161839 → 0.2063632: Dirichlet walls remove heat by boundary flux, the cubic damping removes λΣu³ΔxΔyΔt per step, and these exceed the small positive source injection ΣSΔxΔyΔt. The maximum |u| also decreases and no oscillation/blow-up appears, confirming dissipative stability. Serialized and overlapped runs give identical traces (parity_max_diff = 0.000e+00 < 10⁻¹²).
